In [ ]:
import imageio
import imageio_ffmpeg
from pathlib import Path

PROJECT_DIR = Path.cwd()
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_DIR = PROJECT_DIR / "input_mp4"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
print("Using ffmpeg backend at:", ffmpeg_path)

for webm in INPUT_DIR.glob("*.webm"):
    mp4 = OUTPUT_DIR / (webm.stem + ".mp4")
    print(f"Converting {webm.name} -> {mp4.name}")

    reader = imageio.get_reader(webm)
    fps = reader.get_meta_data().get("fps", 30)

    writer = imageio.get_writer(
        mp4,
        fps=fps,
        codec="libx264",
        pixelformat="yuv420p"
    )

    for frame in reader:
        writer.append_data(frame)

    reader.close()
    writer.close()

print("✅ Done converting all videos")

In [ ]:
import os
import shutil
import math

# ====== CONFIG ======
source_dir = "input_mp4_full"
num_splits = 40
output_prefix = "input_part_"
# ====================

# Lấy danh sách file mp4
files = [f for f in os.listdir(source_dir) if f.lower().endswith(".mp4")]
files.sort()

total_files = len(files)
files_per_split = math.ceil(total_files / num_splits)

print(f"Total files: {total_files}")
print(f"Files per split: {files_per_split}")

for i in range(num_splits):
    start = i * files_per_split
    end = start + files_per_split
    split_files = files[start:end]

    if not split_files:
        break

    split_folder = f"{output_prefix}{i+1}"
    os.makedirs(split_folder, exist_ok=True)

    for file in split_files:
        src_path = os.path.join(source_dir, file)
        dst_path = os.path.join(split_folder, file)
        shutil.move(src_path, dst_path)   # dùng move (cắt)
        # nếu muốn copy thì dùng shutil.copy2()

    print(f"Created {split_folder} with {len(split_files)} files")

print("Done!")
